# E-Commerce Sales Analytics — Data Cleaning & Preparation
**Phase 2: Data Preparation (Python/Pandas)**

This notebook loads the 3 raw CSVs, cleans them, merges them into one fact table, and exports the cleaned files ready for SQL Server / Power BI / Tableau.

## Step 1: Import libraries & load raw data

In [13]:
import pandas as pd
import numpy as np

orders = pd.read_csv('List of Orders.csv')
details = pd.read_csv('Order Details.csv')
target = pd.read_csv('Sales target.csv')

print('Orders:', orders.shape)
print('Order Details:', details.shape)
print('Sales Target:', target.shape)

Orders: (560, 5)
Order Details: (1500, 6)
Sales Target: (36, 3)


## Step 2: Inspect the data (nulls, dtypes, duplicates)

In [14]:
orders.info()
print('\nMissing values:\n', orders.isna().sum())
print('\nDuplicate rows:', orders.duplicated().sum())
orders.tail(10)  # check for blank trailing rows

<class 'pandas.DataFrame'>
RangeIndex: 560 entries, 0 to 559
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Order ID      500 non-null    str  
 1   Order Date    500 non-null    str  
 2   CustomerName  500 non-null    str  
 3   State         500 non-null    str  
 4   City          500 non-null    str  
dtypes: str(5)
memory usage: 42.1 KB

Missing values:
 Order ID        60
Order Date      60
CustomerName    60
State           60
City            60
dtype: int64

Duplicate rows: 59


,Order ID,Order Date,CustomerName,State,City
550,NaN,NaN,NaN,NaN,NaN
551,NaN,NaN,NaN,NaN,NaN
552,NaN,NaN,NaN,NaN,NaN
553,NaN,NaN,NaN,NaN,NaN
554,NaN,NaN,NaN,NaN,NaN
555,NaN,NaN,NaN,NaN,NaN
556,NaN,NaN,NaN,NaN,NaN
557,NaN,NaN,NaN,NaN,NaN
558,NaN,NaN,NaN,NaN,NaN
559,NaN,NaN,NaN,NaN,NaN


In [15]:
details.info()
print('\nMissing values:\n', details.isna().sum())
print('\nDuplicate rows:', details.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Order ID      1500 non-null   str    
 1   Amount        1500 non-null   float64
 2   Profit        1500 non-null   float64
 3   Quantity      1500 non-null   int64  
 4   Category      1500 non-null   str    
 5   Sub-Category  1500 non-null   str    
dtypes: float64(2), int64(1), str(3)
memory usage: 104.9 KB

Missing values:
 Order ID        0
Amount          0
Profit          0
Quantity        0
Category        0
Sub-Category    0
dtype: int64

Duplicate rows: 0


In [16]:
target.info()
print('\nMissing values:\n', target.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Month of Order Date  36 non-null     str    
 1   Category             36 non-null     str    
 2   Target               36 non-null     float64
dtypes: float64(1), str(2)
memory usage: 1.5 KB

Missing values:
 Month of Order Date    0
Category               0
Target                 0
dtype: int64


## Step 3: Clean `Orders`
- Drop fully-blank trailing rows (found 60 in the raw file)
- Trim whitespace from text columns (e.g. `"Kerala "` -> `"Kerala"`)
- Convert `Order Date` to a proper datetime

In [17]:
orders = orders.dropna(subset=['Order ID']).copy()

for col in ['CustomerName', 'State', 'City']:
    orders[col] = orders[col].str.strip()

orders['Order Date'] = pd.to_datetime(orders['Order Date'], format='%d-%m-%Y')

print('Cleaned Orders shape:', orders.shape)
print('Duplicates after cleaning:', orders.duplicated().sum())
orders.head()

Cleaned Orders shape: (500, 5)
Duplicates after cleaning: 0


,Order ID,Order Date,CustomerName,State,City
0,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad
1,B-25602,2018-04-01,Pearl,Maharashtra,Pune
2,B-25603,2018-04-03,Jahan,Madhya Pradesh,Bhopal
3,B-25604,2018-04-03,Divsha,Rajasthan,Jaipur
4,B-25605,2018-04-05,Kasheen,West Bengal,Kolkata


## Step 4: Clean `Order_Details`
- Trim whitespace from Category / Sub-Category
- Check for negative Amount or Quantity (data-quality check)

In [18]:
details['Category'] = details['Category'].str.strip()
details['Sub-Category'] = details['Sub-Category'].str.strip()

print('Negative Amount rows:', (details['Amount'] < 0).sum())
print('Negative/zero Quantity rows:', (details['Quantity'] <= 0).sum())
details.head()

Negative Amount rows: 0
Negative/zero Quantity rows: 0


,Order ID,Amount,Profit,Quantity,Category,Sub-Category
0,B-25601,1275.0,-1148.0,7,Furniture,Bookcases
1,B-25601,66.0,-12.0,5,Clothing,Stole
2,B-25601,8.0,-2.0,3,Clothing,Hankerchief
3,B-25601,80.0,-56.0,4,Electronics,Electronic Games
4,B-25602,168.0,-111.0,2,Electronics,Phones


## Step 5: Clean `Sales_Target`
- Trim whitespace, parse `Month of Order Date` (format `Apr-18`)

In [19]:
target['Category'] = target['Category'].str.strip()
target['Month of Order Date'] = pd.to_datetime(target['Month of Order Date'], format='%b-%y')
target.head()

,Month of Order Date,Category,Target
0,2018-04-01,Furniture,10400.0
1,2018-05-01,Furniture,10500.0
2,2018-06-01,Furniture,10600.0
3,2018-07-01,Furniture,10800.0
4,2018-08-01,Furniture,10900.0


## Step 6: Referential integrity check
Confirm every `Order ID` in Order_Details exists in Orders (and vice versa).

In [20]:
orphan_details = set(details['Order ID']) - set(orders['Order ID'])
orphan_orders  = set(orders['Order ID']) - set(details['Order ID'])

print('Order_Details rows with no matching Order:', len(orphan_details))
print('Orders with no matching Order_Details:', len(orphan_orders))

Order_Details rows with no matching Order: 0
Orders with no matching Order_Details: 0


## Step 7: Merge into one Fact table + calculated columns
- `Profit Margin %` = Profit / Amount
- `Order Month` = month bucket for trend charts
- `Is Loss Making` = flag for negative-profit line items
- `Unit Price` = Amount / Quantity

In [21]:
fact = details.merge(orders, on='Order ID', how='left')

fact['Profit Margin %'] = (fact['Profit'] / fact['Amount'] * 100).round(2)
fact['Order Month'] = fact['Order Date'].dt.to_period('M').dt.to_timestamp()
fact['Order Month Name'] = fact['Order Date'].dt.strftime('%b-%Y')
fact['Order Year'] = fact['Order Date'].dt.year
fact['Is Loss Making'] = fact['Profit'] < 0
fact['Unit Price'] = (fact['Amount'] / fact['Quantity']).round(2)

print('Fact table shape:', fact.shape)
fact.head()

Fact table shape: (1500, 16)


,Order ID,Amount,Profit,Quantity,Category,Sub-Category,Order Date,CustomerName,State,City,Profit Margin %,Order Month,Order Month Name,Order Year,Is Loss Making,Unit Price
0,B-25601,1275.0,-1148.0,7,Furniture,Bookcases,2018-04-01,Bharat,Gujarat,Ahmedabad,-90.04,2018-04-01,Apr-2018,2018,True,182.14
1,B-25601,66.0,-12.0,5,Clothing,Stole,2018-04-01,Bharat,Gujarat,Ahmedabad,-18.18,2018-04-01,Apr-2018,2018,True,13.20
2,B-25601,8.0,-2.0,3,Clothing,Hankerchief,2018-04-01,Bharat,Gujarat,Ahmedabad,-25.00,2018-04-01,Apr-2018,2018,True,2.67
3,B-25601,80.0,-56.0,4,Electronics,Electronic Games,2018-04-01,Bharat,Gujarat,Ahmedabad,-70.00,2018-04-01,Apr-2018,2018,True,20.00
4,B-25602,168.0,-111.0,2,Electronics,Phones,2018-04-01,Pearl,Maharashtra,Pune,-66.07,2018-04-01,Apr-2018,2018,True,84.00


## Step 8: Quick sanity-check KPIs
(Same numbers you should see later in Power BI / Tableau — use these to verify your dashboard is correct.)

In [22]:
total_sales = fact['Amount'].sum()
total_profit = fact['Profit'].sum()
total_orders = fact['Order ID'].nunique()
total_customers = fact['CustomerName'].nunique()
avg_order_value = total_sales / total_orders
profit_margin = total_profit / total_sales * 100
total_qty = fact['Quantity'].sum()
return_rate = (fact['Is Loss Making'].sum() / len(fact)) * 100

print(f'Total Sales:        Rs.{total_sales:,.2f}')
print(f'Total Profit:       Rs.{total_profit:,.2f}')
print(f'Total Orders:       {total_orders}')
print(f'Total Customers:    {total_customers}')
print(f'Avg Order Value:    Rs.{avg_order_value:,.2f}')
print(f'Profit Margin:      {profit_margin:.2f}%')
print(f'Quantity Sold:      {total_qty}')
print(f'Loss-making line %: {return_rate:.2f}%')

Total Sales:        Rs.431,502.00
Total Profit:       Rs.23,955.00
Total Orders:       500
Total Customers:    332
Avg Order Value:    Rs.863.00
Profit Margin:      5.55%
Quantity Sold:      5615
Loss-making line %: 33.53%


## Step 9: Export cleaned files (for SQL Server / Power BI / Tableau)

In [23]:
orders.to_csv('Orders_Cleaned.csv', index=False)
details.to_csv('Order_Details_Cleaned.csv', index=False)
target.to_csv('Sales_Target_Cleaned.csv', index=False)
fact.to_csv('Fact_Sales_Cleaned.csv', index=False)

print('All cleaned files exported successfully.')

All cleaned files exported successfully.
